In [ ]:
# 单元格1：导入必要的库和设置基础路径
import h5py
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm  # 进度条显示
import datetime
import glob

# 设置基础路径
base_dir = '/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data'
data_path = os.path.join(base_dir, 'DATA/TRAIN38.mat')
output_base_dir = os.path.join(base_dir, 'processed_data')

# 创建输出目录
os.makedirs(output_base_dir, exist_ok=True)

print(f"数据路径: {data_path}")
print(f"输出目录: {output_base_dir}")

In [ ]:
# 单元格2：加载数据并进行初步分析
# 加载TRAIN38.mat文件
f = h5py.File(data_path, 'r')
arrays = {}
for k, v in f.items():
    arrays[k] = np.array(v)
f.close()

# 提取数据
data = arrays['data'].transpose()  # 转置以获取正确的形状
region = arrays['region'].transpose()
prob_idx = arrays['prob_idx'].transpose()

# 数据基本信息
print(f"数据形状: {data.shape}")
print(f"标签形状: {region.shape}")
print(f"病人索引形状: {prob_idx.shape}")

# 分析唯一的病人ID和标签值
unique_prob_idx = np.unique(prob_idx)
print(f"唯一病人ID: {unique_prob_idx}")
print(f"病人总数: {len(unique_prob_idx)}个")

# 分析标签情况
if region.ndim == 2:
    # 如果标签是多列的
    for i in range(region.shape[1]):
        unique_values = np.unique(region[:, i])
        print(f"标签列 {i} 的唯一值: {unique_values}, 类别数: {len(unique_values)}")
else:
    # 如果标签是单列的
    unique_values = np.unique(region)
    print(f"标签的唯一值: {unique_values}, 类别数: {len(unique_values)}")

# 分析每个病人的样本数量
for idx in unique_prob_idx:
    count = np.sum(prob_idx == idx)
    print(f"病人 {idx}: {count}个样本")

In [ ]:
# 单元格3：按照病人ID划分数据集
# 步骤1：数据集划分
# 第38号病人直接进入测试集，再从其他病人中随机选择7个加入测试集，共8个病人

# 设置随机种子确保结果可重现
np.random.seed(42)

# 病人划分
test_patients = [38]  # 第38号病人直接进入测试集
remaining_patients = [i for i in range(1, 38)]  # 剩余37个病人

# 随机选择7个病人加入测试集
additional_test_patients = np.random.choice(remaining_patients, 7, replace=False)
test_patients.extend(additional_test_patients)

# 剩余的30个病人用于训练和验证
train_val_patients = [p for p in remaining_patients if p not in additional_test_patients]

print(f"测试集病人 ({len(test_patients)}个): {sorted(test_patients)}")
print(f"训练和验证集病人 ({len(train_val_patients)}个): {sorted(train_val_patients)}")

# 根据病人ID划分数据
test_indices = np.where(np.isin(prob_idx, test_patients))[0]
train_val_indices = np.where(np.isin(prob_idx, train_val_patients))[0]

# 提取测试集
test_data = data[test_indices]
test_regions = region[test_indices]
test_prob_idx = prob_idx[test_indices]

# 提取训练和验证集
train_val_data = data[train_val_indices]
train_val_regions = region[train_val_indices]
train_val_prob_idx = prob_idx[train_val_indices]

print(f"测试集样本数: {len(test_data)}")
print(f"训练和验证集样本数: {len(train_val_data)}")

# 保存病人分配信息
patient_allocation = {
    "test_patients": sorted(test_patients),
    "train_val_patients": sorted(train_val_patients)
}

# 释放内存
del data, region, prob_idx, arrays

In [ ]:
# 单元格4：按标签分组处理数据
# 功能：将训练和验证数据按标签分组，随机打乱，并按6:2比例拆分

# 创建输出目录
train_dir = os.path.join(output_base_dir, 'train')
val_dir = os.path.join(output_base_dir, 'val')
test_dir = os.path.join(output_base_dir, 'test')

for directory in [train_dir, val_dir, test_dir]:
    os.makedirs(directory, exist_ok=True)

# 处理标签
label_data = {}  # 按标签存储数据

# 判断标签是一维还是多维
if train_val_regions.ndim == 1:
    # 标签是一维的
    unique_labels = np.unique(train_val_regions)
    
    for label in unique_labels:
        # 找到该标签的所有样本
        indices = np.where(train_val_regions == label)[0]
        samples = train_val_data[indices]
        
        # 随机打乱
        shuffle_indices = np.random.permutation(len(samples))
        samples = samples[shuffle_indices]
        
        # 按6:2比例拆分
        train_size = int(len(samples) * 0.6)
        train_samples = samples[:train_size]
        val_samples = samples[train_size:]
        
        # 存储
        label_data[int(label)] = {
            "train": train_samples,
            "val": val_samples
        }
        
        print(f"标签 {label}: 总样本 {len(samples)}，训练集 {len(train_samples)}，验证集 {len(val_samples)}")
else:
    # 标签是多维的
    # 找出有效的标签列（包含多个值的列）
    valid_columns = []
    for i in range(train_val_regions.shape[1]):
        if len(np.unique(train_val_regions[:, i])) > 1:
            valid_columns.append(i)
    
    print(f"有效标签列: {valid_columns}")
    
    # 确定使用哪一列作为主标签
    # 这里假设第一个有效列是主标签
    main_label_col = valid_columns[0] if valid_columns else 0
    print(f"使用列 {main_label_col} 作为主标签")
    
    # 按主标签分组
    unique_labels = np.unique(train_val_regions[:, main_label_col])
    
    for label in unique_labels:
        # 找到该标签的所有样本
        indices = np.where(train_val_regions[:, main_label_col] == label)[0]
        samples = train_val_data[indices]
        
        # 随机打乱
        shuffle_indices = np.random.permutation(len(samples))
        samples = samples[shuffle_indices]
        
        # 按6:2比例拆分
        train_size = int(len(samples) * 0.6)
        train_samples = samples[:train_size]
        val_samples = samples[train_size:]
        
        # 存储
        label_data[int(label)] = {
            "train": train_samples,
            "val": val_samples
        }
        
        print(f"标签 {label}: 总样本 {len(samples)}，训练集 {len(train_samples)}，验证集 {len(val_samples)}")

print("数据分组和拆分完成！")

In [ ]:
# 单元格5：创建和保存StandardScaler
# 功能：使用所有训练数据拟合StandardScaler并保存

# 合并所有训练样本
all_train_samples = []
for label, data_dict in label_data.items():
    all_train_samples.append(data_dict["train"])

all_train_samples = np.vstack(all_train_samples)
print(f"用于拟合Scaler的训练样本总数: {len(all_train_samples)}")

# 拟合StandardScaler
scaler = StandardScaler()
scaler.fit(all_train_samples)

# 保存Scaler
scaler_path = os.path.join(output_base_dir, 'data_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"Scaler已拟合并保存到: {scaler_path}")
print(f"Scaler均值形状: {scaler.mean_.shape}")
print(f"Scaler方差形状: {scaler.var_.shape}")

# 释放内存
del all_train_samples

In [ ]:
# 单元格6：处理训练集和验证集数据
# 功能：对训练集和验证集数据应用标准化，并按标签保存

# 处理训练集
print("处理训练集...")
for label, data_dict in tqdm(label_data.items()):
    train_samples = data_dict["train"]
    
    # 标准化
    train_samples_scaled = scaler.transform(train_samples)
    
    # 保存
    output_file = os.path.join(train_dir, f"label_{label}_samples_{len(train_samples)}_voxels.npy")
    np.save(output_file, train_samples_scaled)
    print(f"保存标签 {label} 的 {len(train_samples)} 个训练样本到 {output_file}")

# 处理验证集
print("处理验证集...")
for label, data_dict in tqdm(label_data.items()):
    val_samples = data_dict["val"]
    
    # 标准化
    val_samples_scaled = scaler.transform(val_samples)
    
    # 保存
    output_file = os.path.join(val_dir, f"label_{label}_samples_{len(val_samples)}_voxels.npy")
    np.save(output_file, val_samples_scaled)
    print(f"保存标签 {label} 的 {len(val_samples)} 个验证样本到 {output_file}")

print("训练集和验证集处理完成！")

# 释放内存
del label_data

In [ ]:
# 单元格7：处理测试集数据
# 功能：对测试集数据应用标准化，并按标签和病人保存

# 判断标签是一维还是多维
if test_regions.ndim == 1:
    # 标签是一维的
    unique_labels = np.unique(test_regions)
    
    for label in tqdm(unique_labels, desc="处理测试集"):
        # 找到该标签的所有样本
        indices = np.where(test_regions == label)[0]
        samples = test_data[indices]
        p_idx = test_prob_idx[indices]
        
        # 标准化
        samples_scaled = scaler.transform(samples)
        
        # 保存全部
        output_file = os.path.join(test_dir, f"label_{label}_samples_{len(samples)}_voxels.npy")
        np.save(output_file, samples_scaled)
        print(f"保存标签 {label} 的 {len(samples)} 个测试样本到 {output_file}")
        
        # 按病人分别保存（可选）
        for p_id in np.unique(p_idx):
            p_indices = np.where(p_idx == p_id)[0]
            p_samples = samples_scaled[p_indices]
            
            if len(p_samples) > 0:
                p_output_file = os.path.join(test_dir, f"patient_{p_id}_label_{label}_samples_{len(p_samples)}_voxels.npy")
                np.save(p_output_file, p_samples)
else:
    # 标签是多维的
    # 使用与训练集相同的主标签列
    main_label_col = 0  # 使用单元格4中确定的主标签列
    unique_labels = np.unique(test_regions[:, main_label_col])
    
    for label in tqdm(unique_labels, desc="处理测试集"):
        # 找到该标签的所有样本
        indices = np.where(test_regions[:, main_label_col] == label)[0]
        samples = test_data[indices]
        p_idx = test_prob_idx[indices]
        
        # 标准化
        samples_scaled = scaler.transform(samples)
        
        # 保存全部
        output_file = os.path.join(test_dir, f"label_{label}_samples_{len(samples)}_voxels.npy")
        np.save(output_file, samples_scaled)
        print(f"保存标签 {label} 的 {len(samples)} 个测试样本到 {output_file}")
        
        # 按病人分别保存（可选）
        for p_id in np.unique(p_idx):
            p_indices = np.where(p_idx == p_id)[0]
            p_samples = samples_scaled[p_indices]
            
            if len(p_samples) > 0:
                p_output_file = os.path.join(test_dir, f"patient_{p_id}_label_{label}_samples_{len(p_samples)}_voxels.npy")
                np.save(p_output_file, p_samples)

print("测试集处理完成！")

# 释放内存
del test_data, test_regions, test_prob_idx

In [ ]:
# 单元格8：创建数据集索引文件
# 功能：为每个数据集创建索引文件，便于加载

def create_dataset_index(directory):
    """为指定目录创建数据集索引文件"""
    index_file = os.path.join(directory, "label_index.txt")
    
    with open(index_file, 'w') as f:
        f.write("label_id,voxel_count,filename\n")
        
        # 获取所有不包含"patient_"前缀的.npy文件
        data_files = [file for file in os.listdir(directory) 
                     if file.endswith('.npy') and "samples_" in file and not file.startswith("patient_")]
        
        for file in sorted(data_files, key=lambda x: int(x.split('_')[1])):
            # 从文件名提取信息
            parts = file.split('_')
            label_id = parts[1]
            voxel_count = parts[3]
            
            f.write(f"{label_id},{voxel_count},{file}\n")
    
    print(f"索引文件已创建: {index_file}")
    
    # 如果有按病人分类的文件，也为它们创建索引
    patient_files = [file for file in os.listdir(directory) 
                    if file.endswith('.npy') and file.startswith("patient_")]
    
    if patient_files:
        patient_index_file = os.path.join(directory, "patient_index.txt")
        
        with open(patient_index_file, 'w') as f:
            f.write("patient_id,label_id,voxel_count,filename\n")
            
            for file in sorted(patient_files, key=lambda x: (int(x.split('_')[1]), int(x.split('_')[3]))):
                # 从文件名提取信息
                parts = file.split('_')
                patient_id = parts[1]
                label_id = parts[3]
                voxel_count = parts[5]
                
                f.write(f"{patient_id},{label_id},{voxel_count},{file}\n")
        
        print(f"病人索引文件已创建: {patient_index_file}")

# 为每个数据集创建索引
print("创建数据集索引文件...")
create_dataset_index(train_dir)
create_dataset_index(val_dir)
create_dataset_index(test_dir)

In [ ]:
# 单元格9：创建处理汇总信息文件
# 功能：记录数据处理的详细信息，便于查阅和复现

# 计算各数据集样本总数
def count_samples(directory):
    """计算目录中的样本总数"""
    total = 0
    for file in os.listdir(directory):
        if file.endswith('.npy') and "samples_" in file and not file.startswith("patient_"):
            parts = file.split('_')
            count = int(parts[3])
            total += count
    return total

train_samples = count_samples(train_dir)
val_samples = count_samples(val_dir)
test_samples = count_samples(test_dir)

# 创建汇总信息文件
summary_file = os.path.join(output_base_dir, "processing_summary.txt")

with open(summary_file, 'w') as f:
    f.write("脑体素数据处理汇总信息\n")
    f.write("=" * 50 + "\n\n")
    
    f.write("1. 数据来源\n")
    f.write(f"- 原始数据文件: {data_path}\n\n")
    
    f.write("2. 数据集划分\n")
    f.write(f"- 测试集病人 ({len(patient_allocation['test_patients'])}个): {patient_allocation['test_patients']}\n")
    f.write(f"- 训练和验证集病人 ({len(patient_allocation['train_val_patients'])}个): {patient_allocation['train_val_patients']}\n\n")
    
    f.write("3. 数据集统计\n")
    f.write(f"- 训练集: {train_samples} 个样本\n")
    f.write(f"- 验证集: {val_samples} 个样本\n")
    f.write(f"- 测试集: {test_samples} 个样本\n")
    f.write(f"- 总样本: {train_samples + val_samples + test_samples} 个样本\n\n")
    
    f.write("4. 标准化信息\n")
    f.write(f"- Scaler文件: {os.path.basename(scaler_path)}\n")
    f.write(f"- 特征数量: {len(scaler.mean_)}\n\n")
    
    f.write("5. 标签信息\n")
    f.write(f"- 训练集标签文件: {os.path.join('train', 'label_index.txt')}\n")
    f.write(f"- 验证集标签文件: {os.path.join('val', 'label_index.txt')}\n")
    f.write(f"- 测试集标签文件: {os.path.join('test', 'label_index.txt')}\n\n")
    
    f.write("6. 处理时间\n")
    f.write(f"- 处理完成时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"处理汇总信息已保存到: {summary_file}")

In [ ]:
# 单元格10：创建Scaler使用示例代码
# 功能：提供如何使用保存的Scaler进行数据标准化的示例代码

example_code_file = os.path.join(output_base_dir, "scaler_usage_example.py")

with open(example_code_file, 'w') as f:
    f.write("""# 示例：如何使用保存的Scaler处理新数据
import numpy as np
import pickle
from sklearn.preprocessing import StandardScaler

def load_scaler(scaler_path):
    \"\"\"加载保存的StandardScaler\"\"\"
    with open(scaler_path, 'rb') as f:
        return pickle.load(f)

def standardize_data(data, scaler):
    \"\"\"使用加载的Scaler标准化数据\"\"\"
    return scaler.transform(data)

if __name__ == "__main__":
    # 1. 加载Scaler
    scaler_path = "data_scaler.pkl"  # 替换为实际路径
    scaler = load_scaler(scaler_path)
    
    # 2. 加载需要标准化的数据
    # 示例：从文件加载数据
    # data = np.load("your_data_file.npy")
    # 或者从其他来源获取数据
    data = np.random.rand(100, 341)  # 示例数据，实际使用时替换为真实数据
    
    # 3. 使用Scaler进行标准化
    standardized_data = standardize_data(data, scaler)
    
    # 4. 使用标准化后的数据进行预测或其他操作
    # model.predict(standardized_data)
    
    # 打印信息
    print("数据标准化完成！")
    print(f"原始数据形状: {data.shape}")
    print(f"标准化后数据形状: {standardized_data.shape}")
""")

print(f"Scaler使用示例代码已保存到: {example_code_file}")

In [ ]:
# 单元格11：数据加载辅助函数
# 功能：提供加载处理后数据的辅助函数

data_loader_file = os.path.join(output_base_dir, "data_loader.py")

with open(data_loader_file, 'w') as f:
    f.write("""# 数据加载辅助函数
import numpy as np
import os
import glob
import pickle

def load_index(index_file):
    \"\"\"加载标签索引文件\"\"\"
    index_data = {}
    with open(index_file, 'r') as f:
        # 跳过标题行
        next(f)
        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 3:
                label_id = int(parts[0])
                voxel_count = int(parts[1])
                filename = parts[2]
                index_data[label_id] = {'count': voxel_count, 'filename': filename}
    return index_data

def load_data_by_label(data_dir, label_id=None):
    \"\"\"
    加载指定目录下的数据，可选择按标签过滤
    
    参数:
        data_dir: 数据目录路径
        label_id: 要加载的标签ID，如果为None则加载所有标签
    
    返回:
        data: 加载的数据
        labels: 对应的标签
    \"\"\"
    index_file = os.path.join(data_dir, "label_index.txt")
    index_data = load_index(index_file)
    
    data_list = []
    labels_list = []
    
    if label_id is not None:
        # 只加载指定标签
        if label_id in index_data:
            filename = index_data[label_id]['filename']
            file_path = os.path.join(data_dir, filename)
            
            if os.path.exists(file_path):
                data = np.load(file_path)
                labels = np.full(data.shape[0], label_id)
                
                data_list.append(data)
                labels_list.append(labels)
    else:
        # 加载所有标签
        for label_id, info in index_data.items():
            filename = info['filename']
            file_path = os.path.join(data_dir, filename)
            
            if os.path.exists(file_path):
                data = np.load(file_path)
                labels = np.full(data.shape[0], label_id)
                
                data_list.append(data)
                labels_list.append(labels)
    
    if data_list:
        return np.vstack(data_list), np.concatenate(labels_list)
    else:
        return np.array([]), np.array([])

def load_all_datasets(base_dir):
    \"\"\"
    加载所有数据集
    
    参数:
        base_dir: 基础目录路径，包含train、val、test子目录
    
    返回:
        一个字典，包含训练集、验证集和测试集的数据和标签
    \"\"\"
    train_dir = os.path.join(base_dir, 'train')
    val_dir = os.path.join(base_dir, 'val')
    test_dir = os.path.join(base_dir, 'test')
    
    train_data, train_labels = load_data_by_label(train_dir)
    val_data, val_labels = load_data_by_label(val_dir)
    test_data, test_labels = load_data_by_label(test_dir)
    
    return {
        'train': {'data': train_data, 'labels': train_labels},
        'val': {'data': val_data, 'labels': val_labels},
        'test': {'data': test_data, 'labels': test_labels}
    }

def load_scaler(base_dir):
    \"\"\"加载保存的StandardScaler\"\"\"
    scaler_path = os.path.join(base_dir, 'data_scaler.pkl')
    with open(scaler_path, 'rb') as f:
        return pickle.load(f)

# 使用示例
if __name__ == "__main__":
    # 替换为实际路径
    base_dir = "processed_data"
    
    # 加载所有数据集
    datasets = load_all_datasets(base_dir)
    
    # 打印数据集信息
    for dataset_name, dataset in datasets.items():
        print(f"{dataset_name}集: {dataset['data'].shape[0]} 个样本, {len(np.unique(dataset['labels']))} 个唯一标签")
    
    # 加载Scaler
    scaler = load_scaler(base_dir)
    print(f"Scaler已加载，特征数量: {len(scaler.mean_)}")
""")

print(f"数据加载辅助函数已保存到: {data_loader_file}")

In [ ]:
# 单元格12：验证处理结果
# 功能：验证处理后的数据和Scaler是否正确

print("验证处理结果...")

# 验证输出目录结构
print("1. 验证目录结构:")
for directory in [train_dir, val_dir, test_dir]:
    file_count = len([f for f in os.listdir(directory) if f.endswith('.npy')])
    print(f"  - {os.path.basename(directory)}目录: {file_count} 个.npy文件")

# 验证索引文件
print("\n2. 验证索引文件:")
for directory in [train_dir, val_dir, test_dir]:
    index_file = os.path.join(directory, "label_index.txt")
    if os.path.exists(index_file):
        with open(index_file, 'r') as f:
            line_count = sum(1 for _ in f) - 1  # 减去标题行
        print(f"  - {os.path.basename(directory)}索引文件: {line_count} 个条目")

# 验证Scaler
print("\n3. 验证Scaler:")
try:
    with open(scaler_path, 'rb') as f:
        loaded_scaler = pickle.load(f)
    
    print(f"  - Scaler加载成功, 特征数量: {len(loaded_scaler.mean_)}")
    
    # 从训练数据中获取一些样本用于验证
    train_files = glob.glob(os.path.join(train_dir, "*.npy"))
    if train_files:
        test_data = np.load(train_files[0])[:10]  # 只取10个样本用于测试
        transformed1 = scaler.transform(test_data)
        transformed2 = loaded_scaler.transform(test_data)
        
        is_equal = np.allclose(transformed1, transformed2)
        print(f"  - Scaler验证: {'成功' if is_equal else '失败'}")
except Exception as e:
    print(f"  - Scaler验证出错: {str(e)}")

print("\n处理完成！所有数据已按要求处理并保存。")

In [ ]:
# 单元格13：总结和下一步建议
print("""
================================================================================
                           数据处理完成总结
================================================================================

处理流程概述:
1. 从TRAIN38.mat文件中加载数据
2. 将病人分配到测试集和训练+验证集
3. 将训练+验证集数据按标签分组并随机打乱
4. 按6:2比例将训练+验证集拆分为训练集和验证集
5. 使用训练集数据拟合StandardScaler并保存
6. 对所有数据集应用相同的标准化处理
7. 将处理后的数据按标签分别保存
8. 创建索引文件和辅助函数

输出文件:
- 训练集数据: {}/train/*.npy
- 验证集数据: {}/val/*.npy
- 测试集数据: {}/test/*.npy
- 数据标准化器: {}/data_scaler.pkl
- 处理汇总信息: {}/processing_summary.txt
- Scaler使用示例: {}/scaler_usage_example.py
- 数据加载辅助函数: {}/data_loader.py

下一步建议:
1. 使用data_loader.py加载数据集进行模型训练
2. 训练模型时记录使用的Scaler版本和处理方法
3. 对新数据集使用相同的Scaler进行标准化，确保模型泛化性
4. 考虑对处理后的数据进行可视化分析，验证标准化效果
""".format(
    output_base_dir, output_base_dir, output_base_dir, 
    output_base_dir, output_base_dir, output_base_dir, output_base_dir
))